In [5]:
import pandas as pd
import numpy as np

## Calculate Day-to-Day Precipitation Transition Probability Matrix

In [6]:
# Read cleaned precipitation data and compute day-to-day transition matrix (dry/wet)
precip_data = pd.read_csv("../data/processed/precip_data_cleaned.csv", parse_dates=['DATE'])
# Ensure chronological order
precip_data = precip_data.sort_values('DATE').reset_index(drop=True)
# Create binary column: 0 = dry, 1 = wet (treat HPCP > 0 as wet to be robust)
precip_data['HPCP_bin'] = (precip_data['HPCP'] > 0).astype(int)
# Shift to get next-day value and drop the final NA that has no next day
precip_data['next'] = precip_data['HPCP_bin'].shift(-1)
transitions = precip_data.dropna(subset=['next']).astype({'next':'int'})
# Count transitions: rows = today, columns = next day
transition_counts = transitions.groupby(['HPCP_bin','next']).size().unstack(fill_value=0)
# Convert to probabilities P(next | today) by normalizing each row
transition_prob = transition_counts.div(transition_counts.sum(axis=1), axis=0)
print('Transition counts (rows=today, cols=next day):')
print(transition_counts)
print('\nTransition probabilities P(next | today) (rows=today, cols=next day):')
print(transition_prob)
# Keep results available in the notebook namespace for later use
transition_counts_df = transition_counts
transition_prob_df = transition_prob


Transition counts (rows=today, cols=next day):
next        0   1
HPCP_bin         
0         130  52
1          52  55

Transition probabilities P(next | today) (rows=today, cols=next day):
next             0         1
HPCP_bin                    
0         0.714286  0.285714
1         0.485981  0.514019


In [7]:
# Read AQI data, parse dates, and merge with precipitation data
aqi_data = pd.read_csv("../data/processed/aqi_cleaned.csv", parse_dates=['Date'])

# Merge AQI and precipitation by date and create a 1-6 aqi_category
precip_data['Date'] = pd.to_datetime(precip_data['DATE'])
aqi_data['Date'] = pd.to_datetime(aqi_data['Date'])

precip_aqi_merged = pd.merge(
    aqi_data[['Date', 'AQI']],
    precip_data[['Date', 'HPCP']],
    on='Date',
    how='inner'   # use 'inner' to keep only dates present in both; change to 'left' if you prefer all AQI dates
).sort_values('Date').reset_index(drop=True)

# Map AQI to categories 1-6
bins = [-1, 50, 100, 150, 200, 300, np.inf]
labels = [1, 2, 3, 4, 5, 6]
precip_aqi_merged['aqi_category'] = pd.cut(precip_aqi_merged['AQI'], bins=bins, labels=labels).astype('Int64')

print(precip_aqi_merged.head())


        Date  AQI  HPCP  aqi_category
0 2013-01-01  160     0             4
1 2013-01-06   75     0             2
2 2013-01-06   75     0             2
3 2013-01-06   75     0             2
4 2013-01-06   75     1             2


In [8]:
# Compute P(rain | aqi_category) where rain = HPCP > 0
precip_aqi_merged['rained'] = (precip_aqi_merged['HPCP'] > 0).astype(int)

# Group by category and compute mean and count
stats = precip_aqi_merged.dropna(subset=['aqi_category']).groupby('aqi_category')['rained'].agg(['mean', 'count'])

# Only use the mean if there are at least 5 observations; otherwise set None
rain_prob_by_aqi = {}
for idx, row in stats.iterrows():
    key = int(idx)
    rain_prob_by_aqi[key] = float(row['mean']) if int(row['count']) >= 5 else None

print("P(rain | aqi_category) (None if <5 obs):", rain_prob_by_aqi)

P(rain | aqi_category) (None if <5 obs): {1: 0.40404040404040403, 2: 0.3492063492063492, 3: None, 4: None}
